# Exploratory corpus analysis

Statistics use the same whitespace tokenization as the default model. Punctuation remains attached to tokens.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
from collections import Counter
import math
corpus = Corpus.load(OUT / "processed.jsonl")
runs = [s.split() for s in corpus]
freq = Counter(t for run in runs for t in run)
lengths = pd.Series([len(run) for run in runs], name="tokens")
total = sum(freq.values())
display(pd.Series({"sentences":len(runs), "tokens":total, "vocabulary":len(freq), "type_token_ratio":len(freq)/total, "unigram_entropy_bits":-sum((v/total)*math.log2(v/total) for v in freq.values())}))
display(lengths.describe())
lengths.plot.hist(bins=20, title="Sentence length distribution")
plt.xlabel("Tokens per sentence")
plt.show()

In [ ]:
top = pd.Series(dict(freq.most_common(20))).sort_values()
top.plot.barh(title="Most frequent tokens")
plt.xlabel("Count")
plt.show()
ranked = sorted(freq.values(), reverse=True)
plt.loglog(range(1,len(ranked)+1), ranked, marker=".")
plt.xlabel("Rank")
plt.ylabel("Frequency")
plt.title("Rank-frequency distribution")
plt.show()
pd.DataFrame(freq.most_common(), columns=["token","count"]).to_csv(OUT / "token_frequency.csv", index=False)